In [1]:
import os, time, copy, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
if os.path.basename(os.getcwd()) == 'notes': os.chdir('..')
torch.manual_seed(42); np.random.seed(42)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('工作目录:', os.getcwd(), '| device:', DEV, '| torch', torch.__version__)

工作目录: d:\github\prediction | device: cpu | torch 2.8.0+cpu


In [2]:
# ============================================================
# 数据: NetCDF 9站 → 80/15/5 分割 → 各子集独立预处理 (与 xg_boost(3) 完全一致, 杜绝泄漏)
# LSTM 输入: (N, 168, F) 多变量历史窗 → 输出 (N, 48) 未来 PM2.5 (直接多步, 无递归)
# 每时间步特征 F=29 (参考 xg_boost 特征项的瞬时版, LSTM 自身建模时序, 故省去滞后/滚动):
#   目标站 PM(1) + 其他8站 PM(8) + 气象(DEWP/HUMI/PRES/TEMP/Iws/precip=6)
#   + 额外气象(blh/msdwswrf/O3=3) + 时间周期(hour/month/dow sin·cos=6) + 风向one-hot(5) = 29
# 未来气象协变量: 未来48h 气象+额外(9维) 的 mean+std = 18维, 拼接到解码头
#   (与 xg_boost 的 FUT_COLS 同语义: 未来气象预报可得, 非PM目标, 无泄漏)
# 目标 = 线性 PM2.5 (expm1), 训练统计量标准化; 评估在线性 ug/m³ 空间
# 参考: https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html
# ============================================================
import netCDF4 as nc

N_ST = 9
WEATHER = ['DEWP','HUMI','PRES','TEMP','Iws','precipitation']
EXTRA   = ['blh','msdwswrf','O3']
TIME_COLS = ['hour_sin','hour_cos','month_sin','month_cos','dow_sin','dow_cos']
CBWD_COLS = ['cbwd_cv','cbwd_SE','cbwd_NW','cbwd_SW','cbwd_NE']
TS_COLS = ['pm_ave'] + [f'pm{i}' for i in range(1, N_ST)] + WEATHER + EXTRA + TIME_COLS + CBWD_COLS
WCOLS   = WEATHER + EXTRA                              # 9 维 (用于未来协变量)
F_IN, L, H = len(TS_COLS), 168, 48                     # 输入特征数 / 回看168h / 预测48h

def load_all(nc_path):
    """读取 9站 PM2.5 + 站0 全部气象/额外变量 (变量映射: K→°C, Pa→hPa, m→mm)"""
    f = nc.Dataset(nc_path)
    t = f.variables['time'][:]
    dt = pd.Timestamp(f.variables['time'].units.split('since ')[1]) + pd.to_timedelta(t, unit='h')
    pm = np.asarray(f.variables['PM2.5'][:])
    temp = f.variables['t2m'][:,0]-273.15; dewp = f.variables['d2m'][:,0]-273.15
    pres = f.variables['sp'][:,0]/100.0;  tp = f.variables['tp'][:,0]*1000.0
    u, v = f.variables['u100'][:,0], f.variables['v100'][:,0]
    iws = np.sqrt(u**2+v**2); wdir = np.degrees(np.arctan2(-u,-v))%360
    cbwd = np.where(iws<0.5,'cv',np.where(wdir<90,'NE',np.where(wdir<180,'SE',np.where(wdir<270,'SW','NW')))).astype('<U2')
    a,b = 17.625,243.04
    humi = np.clip(100*np.exp(a*dewp/(dewp+b+1e-10))/np.exp(a*temp/(temp+b+1e-10)),0,100)
    blh = f.variables['blh'][:,0]; swr = f.variables['msdwswrf'][:,0]; o3 = f.variables['O3'][:,0]
    f.close()
    d = {'pm_ave': pm[:,0]}
    for i in range(1,N_ST): d[f'pm{i}'] = pm[:,i]
    d.update({'DEWP':dewp,'HUMI':humi,'PRES':pres,'TEMP':temp,'cbwd':cbwd,'Iws':iws,
              'precipitation':tp,'blh':blh,'msdwswrf':swr,'O3':o3})
    df = pd.DataFrame(d, index=dt); df.index.name='datetime'
    return df.resample('1h').first()

def preprocess(df, forward_only=False):
    """预处理. forward_only=True(测试集)仅前向填充, 不用任何未来值."""
    d = df.copy()
    def fill(s): return s.ffill() if forward_only else s.interpolate(method='time',limit=168).ffill().bfill()
    d['pm_ave'] = np.log1p(fill(d['pm_ave']))
    for i in range(1,N_ST): d[f'pm{i}'] = np.log1p(fill(d[f'pm{i}']))
    cols = WEATHER + EXTRA
    d[cols] = d[cols].ffill() if forward_only else d[cols].interpolate(method='time').ffill().bfill()
    h = d.index.hour.values.astype(float)
    d['hour_sin']=np.sin(2*np.pi*h/24); d['hour_cos']=np.cos(2*np.pi*h/24)
    m = d.index.month.values.astype(float)
    d['month_sin']=np.sin(2*np.pi*m/12); d['month_cos']=np.cos(2*np.pi*m/12)
    dw = d.index.dayofweek.values.astype(float)
    d['dow_sin']=np.sin(2*np.pi*dw/7); d['dow_cos']=np.cos(2*np.pi*dw/7)
    d['cbwd'] = d['cbwd'].ffill().bfill()
    d['cbwd'] = pd.Categorical(d['cbwd'], categories=['cv','SE','NW','SW','NE'])
    d = pd.concat([d.drop('cbwd',axis=1), pd.get_dummies(d['cbwd'],prefix='cbwd').astype(float)], axis=1)
    return d

def future_cov(W, H):
    """未来H步气象摘要. W:(n,k)标准化气象. 返回 (n,2k): 每个起点T的 mean+std(W[T:T+H]).
    用 rolling(H).mean()/std() 实现: rm[j]=mean(W[j-H+1:j+1]); 未来窗[T:T+H]结束于j=T+H-1."""
    df = pd.DataFrame(W); rm = df.rolling(H, min_periods=1).mean().values
    rs = df.rolling(H, min_periods=1).std().fillna(0).values
    out = np.zeros((len(W), 2*W.shape[1]), dtype=np.float32)
    nT = len(W) - H + 1                         # 合法起点数 (T∈[0, n-H])
    out[:nT, :W.shape[1]] = rm[H-1:H-1+nT]      # out[T] = rm[T+H-1]
    out[:nT, W.shape[1]:] = rs[H-1:H-1+nT]
    return out

# 加载 + 分割 + 预处理 (先分割再处理, 测试集前向填充)
df_raw = load_all('data3/dataset_yrd.nc')
n = len(df_raw); n1 = int(n*0.80); n2 = int(n*0.95)
df_tr = preprocess(df_raw.iloc[:n1])
df_va = preprocess(df_raw.iloc[n1:n2])
df_te = preprocess(df_raw.iloc[n2:], forward_only=True)

# 标准化: 仅用训练统计量 (无泄漏)
Xm = df_tr[TS_COLS].values.astype(np.float32)
FEAT_MEAN = Xm.mean(axis=0); FEAT_STD = Xm.std(axis=0) + 1e-8
def std_feats(df): return ((df[TS_COLS].values.astype(np.float32) - FEAT_MEAN) / FEAT_STD)
Ftr, Fva, Fte = std_feats(df_tr), std_feats(df_va), std_feats(df_te)
# 目标 = 线性 PM2.5, 用训练统计量标准化
PMtr = np.expm1(df_tr['pm_ave'].values).astype(np.float32)
Y_MEAN, Y_STD = float(PMtr.mean()), float(PMtr.std()) + 1e-8
def std_y(df): return ((np.expm1(df['pm_ave'].values).astype(np.float32) - Y_MEAN) / Y_STD)
Ytr, Yva, Yte = std_y(df_tr), std_y(df_va), std_y(df_te)
# 未来气象协变量 (基于已标准化的气象列 WCOLS)
Wtr, Wva, Wte = Ftr[:, [TS_COLS.index(c) for c in WCOLS]], Fva[:, [TS_COLS.index(c) for c in WCOLS]], Fte[:, [TS_COLS.index(c) for c in WCOLS]]
Ctr, Cva, Cte = future_cov(Wtr, H), future_cov(Wva, H), future_cov(Wte, H)
FUT_DIM = Ctr.shape[1]
CLIP_HI = float(np.expm1(df_tr['pm_ave'].max()))

# 滑窗索引: 起点T∈[L, n-H], 训练子采样降重叠 (stride), 验证稀疏; 各窗内历史[T-L:T]与未来[T:T+H]同属一子集, 无跨集泄漏
TRAIN_STRIDE, VAL_STRIDE = 3, 6
def win_idx(nlen, stride):
    return np.arange(L, nlen - H + 1, stride, dtype=np.int64)
idx_tr = win_idx(len(df_tr), TRAIN_STRIDE)
idx_va = win_idx(len(df_va), VAL_STRIDE)
# 训练样本权重: 仅放大上四分位(峰值)样本, 其余=1 不失真 (同 xg_boost 思路)
lvl_tr = np.array([Ytr[T:T+H].mean() for T in idx_tr])
q75, q99 = float(np.quantile(lvl_tr, 0.75)), float(np.quantile(lvl_tr, 0.99))
WT_TR = 1.0 + 1.0 * np.clip((lvl_tr - q75) / (q99 - q75 + 1e-8), 0.0, 1.0)

class WinDS(Dataset):
    def __init__(self, feat, futc, y, idx, w=None):
        self.feat, self.futc, self.y, self.idx, self.w = feat, futc, y, idx, w
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        T = int(self.idx[i])
        x = torch.as_tensor(self.feat[T-L:T])            # (L, F_IN) 历史, 用至 T-1
        c = torch.as_tensor(self.futc[T])                # (FUT_DIM,) 未来气象摘要
        y = torch.as_tensor(self.y[T:T+H])               # (H,) 未来 PM (标准化)
        if self.w is not None: return x, c, y, torch.as_tensor(self.w[i], dtype=torch.float32)
        return x, c, y

BS = 256
tr_loader = DataLoader(WinDS(Ftr, Ctr, Ytr, idx_tr, WT_TR), batch_size=BS, shuffle=True)
va_loader = DataLoader(WinDS(Fva, Cva, Yva, idx_va), batch_size=512, shuffle=False)
print(f'训练 {len(df_tr)} | 验证 {len(df_va)} | 测试 {len(df_te)} | 特征/步 {F_IN} | 未来协变量 {FUT_DIM}')
print(f'滑窗样本: 训练 {len(idx_tr)} (stride={TRAIN_STRIDE}) | 验证 {len(idx_va)} (stride={VAL_STRIDE})')
print(f'目标(线性) mean={Y_MEAN:.1f} std={Y_STD:.1f} | 预测上限 {CLIP_HI:.0f} | 权重[{WT_TR.min():.2f},{WT_TR.max():.2f}]')

训练 56102 | 验证 10519 | 测试 3507 | 特征/步 29 | 未来协变量 18
滑窗样本: 训练 18629 (stride=3) | 验证 1718 (stride=6)
目标(线性) mean=34.7 std=27.8 | 预测上限 241 | 权重[1.00,2.00]


In [ ]:
# ============================================================
# 模型: 双向LSTM编码168h历史 → 注意力池化 → 拼接未来气象协变量 → MLP直出48步
# batch_first=True: 输入 (B, L, F); output (B, L, 2*hidden); h_n (layers*2, B, hidden)
# 参考: https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html
# ============================================================
HIDDEN, N_LAYERS, DROP = 96, 2, 0.2

class LSTMForecaster(nn.Module):
    def __init__(self, n_in, hidden, n_layers, fut_dim, n_out, drop=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_in, hidden, n_layers, batch_first=True, bidirectional=True, dropout=drop)
        self.attn = nn.Linear(hidden*2, 1)             # 加性注意力打分
        self.drop = nn.Dropout(drop)
        self.head = nn.Sequential(
            nn.Linear(hidden*2 + fut_dim, 256), nn.GELU(), nn.Dropout(drop),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(drop), nn.Linear(128, n_out))
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x, c):                          # x:(B,L,F) c:(B,fut_dim)
        out, _ = self.lstm(x)                         # (B,L,2*hidden)
        w = torch.softmax(self.attn(out).squeeze(-1), dim=1)   # (B,L) 注意力权重
        ctx = (out * w.unsqueeze(-1)).sum(1)          # (B,2*hidden) 加权池化
        return self.head(torch.cat([self.drop(ctx), c], dim=1))

model = LSTMForecaster(F_IN, HIDDEN, N_LAYERS, FUT_DIM, H, DROP).to(DEV)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)
print(f'参数量: {sum(p.numel() for p in model.parameters()):,}')

def weighted_mse(pred, y, w):
    return (w * ((pred - y)**2).mean(dim=1)).mean()

@torch.no_grad()
def eval_val():
    model.eval(); P, A = [], []
    for xb, cb, yb in va_loader:
        p = model(xb.to(DEV), cb.to(DEV)).cpu().numpy()
        P.append(p); A.append(yb.numpy())
    P = np.concatenate(P) * Y_STD + Y_MEAN; A = np.concatenate(A) * Y_STD + Y_MEAN
    return float(np.sqrt(((P - A)**2).mean()))

EPOCHS, PATIENCE = 25, 5
best_val, best_state, bad = 1e9, None, 0
t0 = time.time()
for ep in range(1, EPOCHS+1):
    model.train(); tl = []
    for xb, cb, yb, wb in tr_loader:
        xb, cb, yb, wb = xb.to(DEV), cb.to(DEV), yb.to(DEV), wb.to(DEV)
        opt.zero_grad(); loss = weighted_mse(model(xb, cb), yb, wb); loss.backward(); opt.step()
        tl.append(loss.item())
    vr = eval_val(); sched.step(vr)
    if vr < best_val - 1e-3:
        best_val, best_state, bad = vr, copy.deepcopy(model.state_dict()), 0
    else:
        bad += 1
    print(f'epoch {ep:2d} | train_loss {np.mean(tl):.4f} | val_RMSE(linear) {vr:.2f} | best {best_val:.2f} | bad {bad}/{PATIENCE} | lr {opt.param_groups[0]["lr"]:.1e}', flush=True)
    if bad >= PATIENCE: print('  early stop'); break
print(f'\n训练完成 {time.time()-t0:.0f}s | 最佳 val RMSE(linear) = {best_val:.2f}')
model.load_state_dict(best_state); model.eval()

# per-horizon 偏差校正 (仅验证集估计, 块对齐 idx_va 顺序)
@torch.no_grad()
def val_pred():
    P, A = [], []
    for xb, cb, yb in va_loader:
        P.append(model(xb.to(DEV), cb.to(DEV)).cpu().numpy()); A.append(yb.numpy())
    return np.concatenate(P)*Y_STD+Y_MEAN, np.concatenate(A)*Y_STD+Y_MEAN
P_va, A_va = val_pred()
BIAS_H = (P_va - A_va).mean(axis=0)                  # (48,) 每个 horizon 的系统偏差
ALPHA = 0.5
print(f'验证 per-horizon bias: h1={BIAS_H[0]:+.2f} h24={BIAS_H[23]:+.2f} h48={BIAS_H[47]:+.2f} (校正系数 α={ALPHA})')

参数量: 413,553


In [ ]:
# ============================================================
# 评估: 36 组, 每组 168h 历史 → 一次预测 48h (直接多步, 无递归)
# 增量无关: 直接输出绝对 PM; 偏差校正减 α·bias_h; 截断[0, CLIP_HI]
# 指标: 分组平均 RMSE(48h|24h) + 池化(MAE/FAC2/R/峰值) + 持久性基线; 画图 2×18
# ============================================================
N_GROUPS, N_PRED = 36, 48
STRIDE = (len(df_te) - L - N_PRED) // (N_GROUPS - 1)    # 适应168h回看
PEAK = 75.0

@torch.no_grad()
def predict_groups():
    P, A = [], []
    for g in range(N_GROUPS):
        T = g * STRIDE + L                            # 预测起点, 历史[T-L:T]用至T-1
        x = torch.as_tensor(Fte[T-L:T]).float().unsqueeze(0).to(DEV)   # (1,L,F)
        c = torch.as_tensor(Cte[T]).float().unsqueeze(0).to(DEV)        # (1,FUT_DIM)
        pred = model(x, c).cpu().numpy().ravel() * Y_STD + Y_MEAN
        pred -= ALPHA * BIAS_H                        # per-horizon 偏差校正
        P.append(np.clip(pred, 0, CLIP_HI)); A.append(np.expm1(df_te['pm_ave'].values[T:T+N_PRED]))
    return P, A

def persist_groups():
    P, A = [], []
    pm = np.expm1(df_te['pm_ave'].values)
    for g in range(N_GROUPS):
        T = g * STRIDE + L; P.append(np.full(N_PRED, pm[T-1])); A.append(pm[T:T+N_PRED])
    return P, A

def group_rmse(P, A, k=None):
    return float(np.mean([np.sqrt(((p[:k]-a[:k])**2).mean()) for p, a in zip(P, A)]))

def pooled(P, A, k, tag):
    p = np.concatenate([x[:k] for x in P]); a = np.concatenate([x[:k] for x in A])
    rmse = np.sqrt(((p-a)**2).mean()); mae = np.abs(p-a).mean()
    fac2 = np.mean((p/(a+1e-6) <= 2) & (a/(p+1e-6) <= 2)); r = np.corrcoef(p, a)[0,1]
    pk = a > PEAK
    prmse = np.sqrt(((p[pk]-a[pk])**2).mean()) if pk.sum() else float('nan')
    pmae = np.abs(p[pk]-a[pk]).mean() if pk.sum() else float('nan')
    print(f'  [{tag}] RMSE={rmse:.2f} MAE={mae:.2f} FAC2={fac2:.3f} R={r:.3f} | 峰值(>{PEAK:.0f})n={int(pk.sum())} RMSE={prmse:.2f} MAE={pmae:.2f}')

P, A = predict_groups()
rmses = [np.sqrt(((p-a)**2).mean()) for p, a in zip(P, A)]
fig, axes = plt.subplots(2, 18, figsize=(60, 12)); axes = axes.flatten()
for g in range(N_GROUPS):
    ax = axes[g]; h = np.arange(1, N_PRED+1)
    ax.plot(h, A[g], lw=0.8, label='actual'); ax.plot(h, P[g], lw=0.8, label='predicted')
    ax.set_title(f'G{g+1} RMSE={rmses[g]:.1f}', fontsize=8)
    ax.legend(fontsize=6); ax.set_xlabel('hour'); ax.set_ylabel('PM2.5')
plt.suptitle(f'LSTM Direct (biLSTM+attn+fut-weather, bias-corrected) — {N_GROUPS} Groups', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

PP, AA = persist_groups()
print('=== 分组平均 RMSE (既定基准) ===')
print(f'  LSTM(直接多步) : 48h={group_rmse(P,A):.2f} | 24h={group_rmse(P,A,24):.2f}')
print(f'  持久性         : 48h={group_rmse(PP,AA):.2f} | 24h={group_rmse(PP,AA,24):.2f}')
print('\n=== 池化综合指标 (LSTM) ===')
pooled(P, A, 48, '48h'); pooled(P, A, 24, '24h')
print('=== 池化综合指标 (持久性) ===')
pooled(PP, AA, 48, '48h'); pooled(PP, AA, 24, '24h')
print('\n=== 各组 RMSE(48h|24h) ===')
for g in range(N_GROUPS):
    r48 = np.sqrt(((P[g]-A[g])**2).mean()); r24 = np.sqrt(((P[g][:24]-A[g][:24])**2).mean())
    print(f'  G{g+1:2d}: {r48:6.2f} | {r24:6.2f}')